# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SameehaSyed05/flyrank-ml-internshipp/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My rule prioritizes content that has strong visibility but shows signs that it needs a refresh. The score combines visibility, freshness risk, position opportunity, and content depth. Higher scores mean the page should be reviewed sooner.

Reason codes:
- stale_visible_page: the page is old and still receives meaningful impressions.
- declining_with_demand: the page is declining while still receiving meaningful impressions.
- thin_visible_page: the page has low word count but still receives meaningful impressions.
- page_one_decay_risk: the page ranks on page one but has room to improve.
Signal checks:
1. Demand/impression decline — CONFIRMED. Among rows with prior demand, 62.3% showed lower impressions in the last 30 days than in the previous 30 days. This supports using declining demand as a refresh signal.

2. Position opportunity — MIXED. 7.7% of rows were in positions 11–20. This provides some optimization opportunity, but the signal is not present broadly enough to treat it as a strong standalone indicator.

In [13]:
from google.colab import userdata
import requests

token = userdata.get("HF_TOKEN")

r = requests.get(
    "https://huggingface.co/api/whoami-v2",
    headers={"Authorization": f"Bearer {token}"}
)

print("Token check:", r.status_code)

if r.status_code == 200:
    print("Token is valid.")
    print("Logged in as:", r.json().get("name"))
else:
    print("Token is NOT being accepted by Hugging Face.")

Token check: 200
Token is valid.
Logged in as: sam992255


In [14]:
import duckdb
from google.colab import userdata

con = duckdb.connect()

hf_token = userdata.get("HF_TOKEN")

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE huggingface,
    TOKEN '{hf_token}'
)
""")

rel = "hf://datasets/FlyRank/internship-warehouse"
query_table = f"{rel}/fact_content_query_90d.parquet"

print("Connected successfully.")

Connected successfully.


In [15]:
print(
    con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        MIN(window_start) AS earliest_window_start,
        MAX(window_start) AS latest_window_start,
        MIN(window_end) AS earliest_window_end,
        MAX(window_end) AS latest_window_end
    FROM read_parquet('{query_table}')
    """).df()
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows earliest_window_start latest_window_start earliest_window_end  \
0     2414248            2026-04-02          2026-04-02          2026-06-30   

  latest_window_end  
0        2026-06-30  


In [16]:


signal_data = con.sql(f"""
WITH content_level AS (
    SELECT
        client_hash_id,
        content_hash_id,
        MAX(impressions_last30) AS impressions_last30,
        MAX(impressions_prev30) AS impressions_prev30,
        MAX(avg_position_last30) AS avg_position_last30,
        MAX(avg_position_prev30) AS avg_position_prev30,
        MAX(content_total_impressions_90d) AS visibility
    FROM read_parquet('{query_table}')
    GROUP BY client_hash_id, content_hash_id
)

SELECT *
FROM content_level
""").df()


# ---------------------------------------------------------
# SIGNAL 1: Demand / volume
# ---------------------------------------------------------

signal_data["demand_bucket"] = "No prior demand"

signal_data.loc[
    (signal_data["impressions_prev30"] > 0) &
    (signal_data["impressions_last30"] < signal_data["impressions_prev30"]),
    "demand_bucket"
] = "Declining"

signal_data.loc[
    (signal_data["impressions_prev30"] > 0) &
    (signal_data["impressions_last30"] == signal_data["impressions_prev30"]),
    "demand_bucket"
] = "Stable"

signal_data.loc[
    (signal_data["impressions_prev30"] > 0) &
    (signal_data["impressions_last30"] > signal_data["impressions_prev30"]),
    "demand_bucket"
] = "Growing"


demand_table = (
    signal_data["demand_bucket"]
    .value_counts()
    .rename_axis("demand_bucket")
    .reset_index(name="n")
)

demand_table["share"] = (
    demand_table["n"] / len(signal_data)
).round(3)

print("SIGNAL 1 — DEMAND / IMPRESSION TREND")
print(demand_table.to_string(index=False))

prior_demand = signal_data[signal_data["impressions_prev30"] > 0]

decline_rate = (
    (prior_demand["impressions_last30"] < prior_demand["impressions_prev30"])
    .mean()
)

print("\nPrior-demand rows:", len(prior_demand))
print("Decline rate:", round(decline_rate, 3))


if decline_rate >= 0.50:
    print("Verdict: CONFIRMED")
elif decline_rate >= 0.25:
    print("Verdict: MIXED")
else:
    print("Verdict: OPPOSITE")



signal_data["position_bucket"] = "Missing"

signal_data.loc[
    signal_data["avg_position_last30"] <= 10,
    "position_bucket"
] = "Top 10"

signal_data.loc[
    (signal_data["avg_position_last30"] > 10) &
    (signal_data["avg_position_last30"] <= 20),
    "position_bucket"
] = "Positions 11-20"

signal_data.loc[
    signal_data["avg_position_last30"] > 20,
    "position_bucket"
] = "Below 20"


position_table = (
    signal_data["position_bucket"]
    .value_counts()
    .rename_axis("position_bucket")
    .reset_index(name="n")
)

position_table["share"] = (
    position_table["n"] / len(signal_data)
).round(3)

print("\nSIGNAL 2 — POSITION OPPORTUNITY")
print(position_table.to_string(index=False))

position_opportunity_rate = (
    signal_data["position_bucket"].eq("Positions 11-20").mean()
)

print("\nRows in positions 11-20:",
      int((signal_data["position_bucket"] == "Positions 11-20").sum()))

print("Position opportunity share:",
      round(position_opportunity_rate, 3))


if position_opportunity_rate >= 0.10:
    print("Verdict: CONFIRMED")
elif position_opportunity_rate > 0:
    print("Verdict: MIXED")
else:
    print("Verdict: FALSE")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

SIGNAL 1 — DEMAND / IMPRESSION TREND
  demand_bucket     n  share
      Declining 74932  0.560
        Growing 42123  0.315
No prior demand 13666  0.102
         Stable  3131  0.023

Prior-demand rows: 120186
Decline rate: 0.623
Verdict: CONFIRMED

SIGNAL 2 — POSITION OPPORTUNITY
position_bucket     n  share
       Below 20 92863  0.694
        Missing 16848  0.126
         Top 10 13827  0.103
Positions 11-20 10314  0.077

Rows in positions 11-20: 10314
Position opportunity share: 0.077
Verdict: MIXED


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [17]:


import os
import pandas as pd

queue = con.sql(f"""
WITH base AS (
    SELECT
        client_hash_id,
        content_hash_id,

        MAX(content_total_impressions_90d) AS visibility,
        MAX(impressions_last30) AS impressions_last30,
        MAX(impressions_prev30) AS impressions_prev30,
        MAX(clicks_last30) AS clicks_last30,
        MAX(clicks_prev30) AS clicks_prev30,
        MAX(avg_position_last30) AS position_last30,
        MAX(avg_position_prev30) AS position_prev30,
        MAX(content_visible_query_count) AS visible_queries
    FROM read_parquet('{query_table}')
    GROUP BY client_hash_id, content_hash_id
)

SELECT
    client_hash_id,
    content_hash_id,

    visibility,
    impressions_last30,
    impressions_prev30,
    clicks_last30,
    clicks_prev30,
    position_last30,
    position_prev30,
    visible_queries,

    -- Score: prioritize visible content showing a decline
    (
        CASE
            WHEN visibility > 0 THEN 40
            ELSE 0
        END
        +
        CASE
            WHEN impressions_prev30 > 0
                 AND impressions_last30 < impressions_prev30
            THEN 30
            ELSE 0
        END
        +
        CASE
            WHEN clicks_prev30 > 0
                 AND clicks_last30 < clicks_prev30
            THEN 20
            ELSE 0
        END
        +
        CASE
            WHEN position_last30 > 10
                 AND position_last30 <= 20
            THEN 10
            ELSE 0
        END
    ) AS score,

    CASE
        WHEN visibility > 0
             AND impressions_prev30 > 0
             AND impressions_last30 < impressions_prev30
        THEN 'declining_with_demand'

        WHEN visibility > 0
             AND position_last30 > 10
             AND position_last30 <= 20
        THEN 'page_one_decay_risk'

        ELSE 'review'
    END AS reason_code,

    CASE
        WHEN visibility > 0
             AND impressions_prev30 > 0
             AND impressions_last30 < impressions_prev30
        THEN 'refresh'

        WHEN visibility > 0
             AND position_last30 > 10
             AND position_last30 <= 20
        THEN 'optimize'

        ELSE 'review'
    END AS action

FROM base
ORDER BY score DESC
""").df()

queue.insert(0, "rank", range(1, len(queue) + 1))

output_path = "work/outputs/baseline_action_score.csv"
os.makedirs("work/outputs", exist_ok=True)

queue.to_csv(output_path, index=False)

print("Queue created successfully.")
print("Rows:", len(queue))
print("Output:", output_path)
print()
print(queue.head(10))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Queue created successfully.
Rows: 133852
Output: work/outputs/baseline_action_score.csv

   rank           client_hash_id           content_hash_id  visibility  \
0     1  client_08a6a72ff48e62c0  content_56175887ec07bc35       11040   
1     2  client_08a6a72ff48e62c0  content_5f2c5708acd38231        3407   
2     3  client_0fa64a184f18a4a0  content_5b2f41e90b437a45      137996   
3     4  client_1a8bf67cad4ee525  content_ff6cd30bb57ff448        1856   
4     5  client_20259bd6705d81d4  content_01f0978212148f97        9706   
5     6  client_20259bd6705d81d4  content_bc6c40f278e1b785      123468   
6     7  client_20259bd6705d81d4  content_cf7a2e6a96160a41        5175   
7     8  client_23a62021009f63c4  content_0d932cc91f10bcd4        2774   
8     9  client_23a62021009f63c4  content_0f38658739b67e38       17648   
9    10  client_23a62021009f63c4  content_7b92bf28ee9d4499        4933   

   impressions_last30  impressions_prev30  clicks_last30  clicks_prev30  \
0                 207

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [18]:
# Section 3: Top-20 review

top20 = queue.head(20).copy()

top20["confidence_note"] = top20.apply(
    lambda r: (
        "High confidence: visible content is showing declining demand."
        if r["reason_code"] == "declining_with_demand"
        else "Medium confidence: page-one position suggests an optimization opportunity."
    ),
    axis=1
)

top20["what_would_make_it_wrong"] = top20.apply(
    lambda r: (
        "The decline may be temporary, seasonal, or caused by a short-term change in search demand."
        if r["reason_code"] == "declining_with_demand"
        else "The position may be based on limited observations or may not represent a stable ranking."
    ),
    axis=1
)

review = top20[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "action",
        "reason_code",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
]

print("Top-20 review:")
print(review.to_string(index=False))

Top-20 review:
 rank          client_hash_id          content_hash_id  action           reason_code                                               confidence_note                                                                   what_would_make_it_wrong
    1 client_08a6a72ff48e62c0 content_56175887ec07bc35 refresh declining_with_demand High confidence: visible content is showing declining demand. The decline may be temporary, seasonal, or caused by a short-term change in search demand.
    2 client_08a6a72ff48e62c0 content_5f2c5708acd38231 refresh declining_with_demand High confidence: visible content is showing declining demand. The decline may be temporary, seasonal, or caused by a short-term change in search demand.
    3 client_0fa64a184f18a4a0 content_5b2f41e90b437a45 refresh declining_with_demand High confidence: visible content is showing declining demand. The decline may be temporary, seasonal, or caused by a short-term change in search demand.
    4 client_1a8bf67cad4ee525 con

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [19]:

weak_picks = queue.tail(20).copy()

print("Weakest 20 picks:")
print(
    weak_picks[
        [
            "rank",
            "client_hash_id",
            "content_hash_id",
            "score",
            "reason_code",
            "action"
        ]
    ].to_string(index=False)
)

print("\nLeakage checks:")


future_columns = [
    col for col in queue.columns
    if "future" in col.lower()
    or "next" in col.lower()
]

print("Future-window columns in queue:", future_columns)


sensitive_name_columns = [
    col for col in queue.columns
    if col in ["client_name", "product_name", "url"]
]

print("Product/name/URL columns in queue:", sensitive_name_columns)

print("\nLeakage check passed:",
      len(future_columns) == 0 and len(sensitive_name_columns) == 0)

Weakest 20 picks:
  rank          client_hash_id          content_hash_id  score reason_code action
133833 client_ff644d8251367cbb content_6aa4a210153da42f     40      review review
133834 client_e547b89c05043229 content_15a46bf70c4c656d     40      review review
133835 client_e547b89c05043229 content_15b36a03d32211ed     40      review review
133836 client_e547b89c05043229 content_15be96d8f8b16df2     40      review review
133837 client_e547b89c05043229 content_15ebb3c3cb50745b     40      review review
133838 client_e547b89c05043229 content_15f86d8269a16b63     40      review review
133839 client_e547b89c05043229 content_160ec03a918a3c7e     40      review review
133840 client_e547b89c05043229 content_161db10b4e34e962     40      review review
133841 client_e547b89c05043229 content_1623ac1091da13a8     40      review review
133842 client_e547b89c05043229 content_1658a3da0f037fc1     40      review review
133843 client_e547b89c05043229 content_1678cf19f0e491ab     40      review revie

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.